# OMNet-V3: Magnification-Aware Multi-Task Fusion Framework for Breast Cancer Histopathology
## Specialized for AMD Radeon RX 9060 XT 16 GB / ROCm / Linux Local Execution

---

### Hardware & Environment Constraints:
- **Target GPU:** AMD Radeon RX 9060 XT (16 GB VRAM, RDNA 4, `gfx1200`)
- **Software Stack:** Linux, ROCm 7.2+ with HIP backend (`torch.device("cuda:0")`)
- **Precision Policy:** Full FP32 precision (`amp_enabled = False`) for guaranteed RDNA 4 workload stability
- **Architecture:** Dual-Branch EfficientNet-B0 (1280-d) + ViT-Tiny/16 (192-d) with Magnification-Aware Adaptive Fusion (MAF) and 64-d scale embedding
- **Evaluation:** 5-Fold Patient-Disjoint Cross-Validation on BreakHis across all magnifications (40X, 100X, 200X, 400X)

# Section 1: Environment Setup & Library Imports

In [ ]:
# ============================================================
# Section 1: Environment Setup & Library Imports
# ============================================================
import os
import sys
import glob
import json
import time
import math
import random
import zipfile
import platform
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    auc
)

from tqdm.auto import tqdm
import gc

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("[OK] Core dependencies imported successfully.")

# Section 2: AMD ROCm / HIP Hardware Capability Verification

In [ ]:
# ============================================================
# Section 2: AMD ROCm / HIP Hardware Capability Verification
# ============================================================
print("=" * 65)
print("  AMD ROCm / GPU Hardware Capability Verification")
print("=" * 65)

if not torch.cuda.is_available():
    print("  [WARNING] ROCm GPU not detected. Running on CPU (training will be slow).")
    DEVICE = torch.device("cpu")
else:
    DEVICE = torch.device("cuda:0")
    props = torch.cuda.get_device_properties(0)
    vram_gib = props.total_memory / (1024 ** 3)
    gcn_arch = getattr(props, "gcnArchName", "gfx1200 (detected)")
    hip_version = getattr(torch.version, "hip", "N/A")
    bf16_supported = torch.cuda.is_bf16_supported() if hasattr(torch.cuda, "is_bf16_supported") else False

    print(f"  Backend        : ROCm / HIP (HIP Version: {hip_version})")
    print(f"  Device         : {torch.cuda.get_device_name(0)}")
    print(f"  Architecture   : {gcn_arch}")
    print(f"  Total VRAM     : {vram_gib:.2f} GiB")
    print(f"  Multiprocessors: {getattr(props, 'multi_processor_count', 'N/A')}")
    print(f"  PyTorch        : {torch.__version__}")
    print(f"  BF16 Support   : {bf16_supported}")
    print(f"  Active Device  : {DEVICE}")
    print("  Precision Mode : FP32 (Full Precision Baseline for ROCm 7.2 Stability)")
print("=" * 65)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        if getattr(torch.version, "hip", None) is None:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

# Section 3: Output Storage Directory Hierarchy

In [ ]:
# ============================================================
# Section 3: Output Storage Directory Hierarchy
# ============================================================
PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "output_v3"

GRADCAM_DIR = OUTPUT_DIR / "gradcam"
SCORECAM_DIR = OUTPUT_DIR / "scorecam"
MISCLASSIFIED_DIR = OUTPUT_DIR / "misclassified"
CORRECT_PRED_DIR = OUTPUT_DIR / "correct_predictions"

for d in [OUTPUT_DIR, GRADCAM_DIR, SCORECAM_DIR, MISCLASSIFIED_DIR, CORRECT_PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for fold_idx in range(5):
    (OUTPUT_DIR / f"fold_{fold_idx}").mkdir(parents=True, exist_ok=True)

def save_figure(fig, filename, dpi=300):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    print(f"  [SAVE] Saved figure: {path}")

print(f"[OK] Run output directory initialized: {OUTPUT_DIR}")

# Section 4: Centralized Configuration Object

In [ ]:
# ============================================================
# Section 4: Centralized Configuration Object (Single Source of Truth)
# ============================================================

CONFIG = {
    # --- Data & Image Pipeline ---
    "dataset_id": "trexbytes/breakhislink",
    "output_dir": str(OUTPUT_DIR),
    "image_size": 224,
    "batch_size": 16,               # Baseline batch size for 16GB dual-branch VRAM
    "num_workers": 0,               # Conservative default: avoids multiprocessing crashes in Linux
    "pin_memory": torch.cuda.is_available(),
    "n_splits": 5,                  # 5-Fold patient-disjoint cross-validation
    "magnification_levels": [40, 100, 200, 400],
    "binary_classes": ["benign", "malignant"],
    "subtype_order": ["A", "F", "PT", "TA", "DC", "LC", "MC", "PC"],
    "subtype_to_index": {"A": 0, "F": 1, "PT": 2, "TA": 3, "DC": 4, "LC": 5, "MC": 6, "PC": 7},
    "binary_mapping": {"A": 0, "F": 0, "PT": 0, "TA": 0, "DC": 1, "LC": 1, "MC": 1, "PC": 1},
    "train_resize": 256,
    "train_crop": 224,
    "val_resize": 256,
    "val_crop": 224,
    "stain_aug_prob": 0.5,

    # --- Dual-Branch Architecture ---
    "cnn_backbone": "efficientnet_b0",
    "vit_backbone": "vit_tiny_patch16_224",
    "fusion_dim": 256,
    "magnification_embedding_dim": 64,
    "dropout": 0.3,

    # --- Training & Stability Specialization ---
    "max_epochs": 30,
    "warmup_epochs": 5,
    "early_stopping_patience": 8,
    "base_lr": 1e-4,
    "head_lr": 5e-4,
    "weight_decay": 1e-4,
    "gradient_clip_norm": 1.0,
    "amp_enabled": False,           # FP32 Full Precision (Prevents AMD GPUVM Page Fault on ROCm 7.2)
    "preferred_dtype": "float32",
    "loss_weights": {"binary": 0.3, "subtype": 0.6, "consistency": 0.1},

    # --- Reproducibility ---
    "seed": 42,
    "smoke_test": False,
}

if CONFIG["smoke_test"]:
    CONFIG["n_splits"] = 2
    CONFIG["max_epochs"] = 2
    CONFIG["warmup_epochs"] = 1

set_seed(CONFIG["seed"])

print("=" * 65)
print("  OMNet-V3 Configuration Parameters")
print("=" * 65)
for k, v in CONFIG.items():
    print(f"  {k:30s}: {v}")
print("=" * 65)

# Section 5: BreakHis Dataset Acquisition & File Parsing via kagglehub

In [ ]:
# ============================================================
# Section 5: BreakHis Dataset Acquisition & File Parsing
# ============================================================
import kagglehub

dataset_id = CONFIG["dataset_id"]
print(f"[INFO] Acquiring dataset '{dataset_id}' via kagglehub / local cache...")
try:
    download_path = Path(kagglehub.dataset_download(dataset_id))
    print(f"[INFO] Dataset located at: {download_path}")
except Exception as e:
    print(f"[WARNING] kagglehub download error: {e}. Checking local fallback folders...")
    download_path = PROJECT_ROOT / "data_breakhis"

# Unpack any ZIP archives if present
local_extract_dir = PROJECT_ROOT / "data_breakhis"
zip_files = list(download_path.rglob("*.zip")) if download_path.exists() else []

if zip_files:
    local_extract_dir.mkdir(parents=True, exist_ok=True)
    for zf in zip_files:
        print(f"[INFO] Extracting archive {zf.name} to {local_extract_dir}...")
        with zipfile.ZipFile(zf, "r") as z:
            z.extractall(local_extract_dir)
    search_root = local_extract_dir
else:
    search_root = download_path if download_path.exists() else PROJECT_ROOT

print(f"[INFO] Scanning {search_root} for BreakHis PNG image files...")

def parse_breakhis_filename(path: str) -> Optional[Dict[str, Any]]:
    filename = os.path.basename(path)
    if not filename.lower().endswith(".png"):
        return None

    fname_no_ext = filename[:-4]
    parts = fname_no_ext.split("-")

    if len(parts) < 4:
        return None

    try:
        prefix_parts = parts[0].split("_")
        if len(prefix_parts) < 3:
            return None

        raw_class = prefix_parts[1]
        raw_subtype = prefix_parts[2]

        mag_token = parts[-2]
        magnification = int(mag_token)

        if magnification not in CONFIG["magnification_levels"]:
            return None

        class_name = "benign" if raw_class == "B" else "malignant"
        subtype = raw_subtype

        if subtype not in CONFIG["subtype_to_index"]:
            return None

        case_id = "-".join(parts[1:-2])
        patient_id = f"{raw_subtype}_{case_id}"
        seq = int(parts[-1])

        return {
            "file_path": path,
            "filename": filename,
            "class_name": class_name,
            "subtype": subtype,
            "subtype_index": CONFIG["subtype_to_index"][subtype],
            "binary_label": CONFIG["binary_mapping"][subtype],
            "magnification": magnification,
            "magnification_index": CONFIG["magnification_levels"].index(magnification),
            "patient_id": patient_id,
            "sequence": seq,
        }
    except Exception:
        return None

records = []
for full_path in search_root.rglob("*.png"):
    parsed = parse_breakhis_filename(str(full_path.resolve()))
    if parsed is not None:
        records.append(parsed)

if len(records) == 0:
    # Check alternative local dataset path
    for candidate in [PROJECT_ROOT / "raw_data" / "breakhis", PROJECT_ROOT / "BreakHis"]:
        if candidate.exists():
            for full_path in candidate.rglob("*.png"):
                parsed = parse_breakhis_filename(str(full_path.resolve()))
                if parsed is not None:
                    records.append(parsed)
        if len(records) > 0:
            break

assert len(records) > 0, f"[ERROR] No valid BreaKHis images found in {search_root}!"

metadata = pd.DataFrame(records)
print(f"[OK] Discovered {len(metadata)} total images across {metadata['patient_id'].nunique()} patients.")

print("\n--- Distribution by Binary Class ---")
print(metadata["class_name"].value_counts())
print("\n--- Distribution by Subtype ---")
print(metadata["subtype"].value_counts())
print("\n--- Distribution by Magnification ---")
print(metadata["magnification"].value_counts())

# Section 6: Patient-Disjoint 5-Fold Splitting (StratifiedGroupKFold)

In [ ]:
# ============================================================
# Section 6: Patient-Disjoint 5-Fold Splitting
# ============================================================
sgkf = StratifiedGroupKFold(n_splits=CONFIG["n_splits"], shuffle=True, random_state=CONFIG["seed"])

split_indices = []
for train_idx, test_idx in sgkf.split(metadata, metadata["subtype_index"], metadata["patient_id"]):
    split_indices.append((train_idx, test_idx))

# Verify strict patient disjointness
for fold, (train_idx, test_idx) in enumerate(split_indices):
    train_p = set(metadata.iloc[train_idx]["patient_id"])
    test_p = set(metadata.iloc[test_idx]["patient_id"])
    overlap = len(train_p & test_p)
    assert overlap == 0, f"[LEAKAGE] Fold {fold} has {overlap} overlapping patients!"

print(f"[PASS] 5-Fold Patient-Disjoint Splits verified with zero patient overlap across folds.")

# Section 7: Stain Augmentation & Preprocessing Pipelines

In [ ]:
# ============================================================
# Section 7: Stain Augmentation & Preprocessing Pipelines
# ============================================================

def macenko_stain_perturbation(image: Image.Image, alpha_range=(0.8, 1.2), beta_range=(-0.1, 0.1)) -> Image.Image:
    """
    Performs Macenko-style optical density stain perturbation on CPU.
    Perturbs H&E stain concentration vectors to simulate lab staining variations.
    """
    img_np = np.array(image).astype(np.float32)
    if img_np.ndim != 3 or img_np.shape[2] != 3:
        return image

    I = img_np.reshape((-1, 3))
    # Convert RGB to Optical Density (OD)
    I_nonzero = np.maximum(I, 1.0)
    OD = -np.log10(I_nonzero / 255.0)

    # Remove transparent/background pixels (OD < 0.15)
    ODhat = OD[~np.any(OD < 0.15, axis=1)]
    if len(ODhat) < 100:
        return image

    try:
        # SVD decomposition for stain vectors
        _, _, V = np.linalg.svd(ODhat, full_matrices=False)
        That = ODhat.dot(V[:2].T)
        phi = np.arctan2(That[:, 1], That[:, 0])

        min_phi = np.percentile(phi, 1)
        max_phi = np.percentile(phi, 99)

        v1 = V[:2].T.dot(np.array([np.cos(min_phi), np.sin(min_phi)]))
        v2 = V[:2].T.dot(np.array([np.cos(max_phi), np.sin(max_phi)]))

        if v1[0] > v2[0]:
            HE = np.array([v1, v2])
        else:
            HE = np.array([v2, v1])

        # Normalize stain matrix
        HE = HE / np.linalg.norm(HE, axis=1, keepdims=True)

        # Deconvolve stain concentrations
        C = np.linalg.lstsq(HE.T, OD.T, rcond=None)[0]

        # Apply random stain perturbation
        alpha = np.random.uniform(alpha_range[0], alpha_range[1], size=(2, 1))
        beta = np.random.uniform(beta_range[0], beta_range[1], size=(2, 1))
        C_perturbed = C * alpha + beta

        # Reconstruct image from perturbed optical density
        OD_perturbed = HE.T.dot(C_perturbed).T
        I_perturbed = 255.0 * np.power(10.0, -OD_perturbed)
        I_perturbed = np.clip(I_perturbed.reshape(img_np.shape), 0, 255).astype(np.uint8)
        return Image.fromarray(I_perturbed)
    except Exception:
        return image

class BreakHisDataset(Dataset):
    def __init__(self, df: pd.DataFrame, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

        # Standard ImageNet normalization
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        if self.is_train:
            self.spatial_transforms = transforms.Compose([
                transforms.Resize((CONFIG["train_resize"], CONFIG["train_resize"])),
                transforms.RandomCrop(CONFIG["train_crop"]),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.RandomRotation(degrees=15),
                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.0),
                transforms.ToTensor(),
            ])
        else:
            self.spatial_transforms = transforms.Compose([
                transforms.Resize((CONFIG["val_resize"], CONFIG["val_resize"])),
                transforms.CenterCrop(CONFIG["val_crop"]),
                transforms.ToTensor(),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = row["file_path"]

        with Image.open(img_path) as img:
            image = img.convert("RGB")

        # Stain perturbation applied on CPU with probability
        if self.is_train and random.random() < CONFIG["stain_aug_prob"]:
            image = macenko_stain_perturbation(image)

        tensor_img = self.normalize(self.spatial_transforms(image))

        return {
            "image": tensor_img,
            "binary_label": torch.tensor(row["binary_label"], dtype=torch.long),
            "subtype_label": torch.tensor(row["subtype_index"], dtype=torch.long),
            "magnification_index": torch.tensor(row["magnification_index"], dtype=torch.long),
            "magnification": row["magnification"],
            "patient_id": row["patient_id"],
            "file_path": img_path
        }

def build_dataloaders(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    train_ds = BreakHisDataset(train_df, is_train=True)
    val_ds = BreakHisDataset(val_df, is_train=False)
    test_ds = BreakHisDataset(test_df, is_train=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        pin_memory=CONFIG["pin_memory"],
        drop_last=True
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=CONFIG["pin_memory"]
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=CONFIG["pin_memory"]
    )
    return train_loader, val_loader, test_loader

print("[OK] Dataset class and DataLoader pipeline initialized.")

# Section 8: OMNet-V3 Architecture (Magnification-Aware Adaptive Fusion)

In [ ]:
# ============================================================
# Section 8: OMNet-V3 Architecture
# ============================================================

class MagnificationAwareFusion(nn.Module):
    """
    Magnification-Aware Adaptive Fusion (MAF) Module:
    Dynamically balances CNN local morphology and ViT global context
    conditioned on a 64-dimensional magnification scale embedding.
    """
    def __init__(self, feat_dim: int = 256, mag_dim: int = 64):
        super().__init__()
        self.magnification_embedding = nn.Embedding(4, mag_dim)
        self.gate = nn.Sequential(
            nn.Linear(feat_dim * 2 + mag_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.fusion_fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.LayerNorm(feat_dim),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"])
        )

    def forward(self, f_cnn: torch.Tensor, f_vit: torch.Tensor, mag_idx: torch.Tensor):
        e_mag = self.magnification_embedding(mag_idx)
        gate_input = torch.cat([f_cnn, f_vit, e_mag], dim=1)
        alpha = self.gate(gate_input)  # (B, 1)

        # Gated convex combination of representations
        f_combined = alpha * f_cnn + (1.0 - alpha) * f_vit
        f_out = self.fusion_fc(f_combined)
        return f_out, alpha


class OMNetV3(nn.Module):
    """
    Master OMNet-V3 Architecture:
    EfficientNet-B0 (1280-d) + ViT-Tiny/16 (192-d) -> Linear Projections (256-d) -> MAF Module -> Multi-Task Heads
    """
    def __init__(self):
        super().__init__()
        # 1. Local Branch: EfficientNet-B0
        self.cnn_backbone = timm.create_model("efficientnet_b0", pretrained=True, num_classes=0)
        cnn_feat_dim = self.cnn_backbone.num_features  # 1280

        # 2. Global Branch: ViT-Tiny/16
        self.vit_backbone = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=0)
        vit_feat_dim = self.vit_backbone.num_features  # 192

        # 3. Projection Layers to 256-d shared space
        self.cnn_proj = nn.Sequential(
            nn.Linear(cnn_feat_dim, CONFIG["fusion_dim"]),
            nn.LayerNorm(CONFIG["fusion_dim"]),
            nn.ReLU(inplace=True),
            nn.Dropout(CONFIG["dropout"])
        )
        self.vit_proj = nn.Sequential(
            nn.Linear(vit_feat_dim, CONFIG["fusion_dim"]),
            nn.LayerNorm(CONFIG["fusion_dim"]),
            nn.ReLU(inplace=True),
            nn.Dropout(CONFIG["dropout"])
        )

        # 4. Magnification-Aware Adaptive Fusion (MAF)
        self.fusion = MagnificationAwareFusion(
            feat_dim=CONFIG["fusion_dim"],
            mag_dim=CONFIG["magnification_embedding_dim"]
        )

        # 5. Multi-Task Classification Heads
        self.binary_head = nn.Linear(CONFIG["fusion_dim"], 2)
        self.subtype_head = nn.Linear(CONFIG["fusion_dim"], 8)

    def forward(self, x: torch.Tensor, mag_idx: torch.Tensor) -> Dict[str, torch.Tensor]:
        # Branch 1: CNN forward pass
        f_cnn_raw = self.cnn_backbone(x)
        f_cnn = self.cnn_proj(f_cnn_raw)

        # Branch 2: ViT forward pass
        f_vit_raw = self.vit_backbone(x)
        f_vit = self.vit_proj(f_vit_raw)

        # Branch 3: MAF Fusion
        f_out, alpha = self.fusion(f_cnn, f_vit, mag_idx)

        # Output Logits
        binary_logits = self.binary_head(f_out)
        subtype_logits = self.subtype_head(f_out)

        return {
            "binary_logits": binary_logits,
            "subtype_logits": subtype_logits,
            "f_out": f_out,
            "alpha": alpha,
            "f_cnn": f_cnn,
            "f_vit": f_vit
        }

print("[OK] OMNet-V3 neural network architecture defined.")

# Section 9: Hierarchical Multi-Task Loss Function

In [ ]:
# ============================================================
# Section 9: Hierarchical Multi-Task Loss Function
# ============================================================

def compute_class_weights(df: pd.DataFrame) -> Tuple[torch.Tensor, torch.Tensor]:
    bin_counts = df["binary_label"].value_counts().sort_index().values
    bin_weights = torch.tensor(len(df) / (2.0 * np.maximum(bin_counts, 1)), dtype=torch.float32)

    sub_counts = df["subtype_index"].value_counts().sort_index().values
    sub_weights = torch.tensor(len(df) / (8.0 * np.maximum(sub_counts, 1)), dtype=torch.float32)
    return bin_weights / bin_weights.mean(), sub_weights / sub_weights.mean()


class HierarchicalLoss(nn.Module):
    """
    Hierarchical multi-task loss combining:
    1. Weighted Binary Cross-Entropy Loss (Benign vs Malignant)
    2. Weighted Subtype Cross-Entropy Loss (8-Subtype Classification)
    3. Consistency Regularization Loss (enforces agreement between subtype and binary probabilities)
    """
    def __init__(self, class_weights_binary: torch.Tensor, class_weights_subtype: torch.Tensor):
        super().__init__()
        self.loss_binary = nn.CrossEntropyLoss(weight=class_weights_binary)
        self.loss_subtype = nn.CrossEntropyLoss(weight=class_weights_subtype)

    def forward(self, binary_logits, subtype_logits, binary_targets, subtype_targets):
        L_binary = self.loss_binary(binary_logits, binary_targets)
        L_subtype = self.loss_subtype(subtype_logits, subtype_targets)

        # Consistency loss: Subtype probabilities mapped to binary grouping
        subtype_probs = torch.softmax(subtype_logits, dim=1)
        p_benign_from_subtype = subtype_probs[:, :4].sum(dim=1)
        p_malignant_from_subtype = subtype_probs[:, 4:].sum(dim=1)
        aggregated_binary_probs = torch.stack([p_benign_from_subtype, p_malignant_from_subtype], dim=1)

        binary_probs = torch.softmax(binary_logits, dim=1)
        L_consistency = F.mse_loss(binary_probs, aggregated_binary_probs)

        w = CONFIG["loss_weights"]
        L_total = w["binary"] * L_binary + w["subtype"] * L_subtype + w["consistency"] * L_consistency
        return L_total, {"binary": L_binary.item(), "subtype": L_subtype.item(), "consistency": L_consistency.item()}

print("[OK] Hierarchical multi-task loss configured.")

# Section 10: Training Engine (FP32 ROCm Optimized)

In [ ]:
# ============================================================
# Section 10: Training Engine (FP32 ROCm Optimized)
# ============================================================

def set_requires_grad(model, flag):
    for p in model.parameters():
        p.requires_grad = flag

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    preds_binary, preds_subtype = [], []
    labels_binary, labels_subtype = [], []

    for batch in tqdm(loader, leave=False, desc="Train Epoch"):
        images = batch["image"].to(device, non_blocking=True)
        binary_targets = batch["binary_label"].to(device, non_blocking=True)
        subtype_targets = batch["subtype_label"].to(device, non_blocking=True)
        magnification_idx = batch["magnification_index"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # FP32 stable dual-backbone execution
        outputs = model(images, magnification_idx)
        loss, _ = criterion(
            outputs["binary_logits"],
            outputs["subtype_logits"],
            binary_targets,
            subtype_targets,
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Encountered non-finite loss: {loss.item()}")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip_norm"])
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds_binary.append(torch.argmax(outputs["binary_logits"], dim=1).detach().cpu())
        preds_subtype.append(torch.argmax(outputs["subtype_logits"], dim=1).detach().cpu())
        labels_binary.append(binary_targets.cpu())
        labels_subtype.append(subtype_targets.cpu())

    n_samples = max(len(loader.dataset), 1)
    epoch_loss = running_loss / n_samples
    binary_pred = torch.cat(preds_binary).numpy()
    subtype_pred = torch.cat(preds_subtype).numpy()
    binary_true = torch.cat(labels_binary).numpy()
    subtype_true = torch.cat(labels_subtype).numpy()

    return {
        "loss": epoch_loss,
        "binary_accuracy": float(accuracy_score(binary_true, binary_pred)),
        "subtype_accuracy": float(accuracy_score(subtype_true, subtype_pred)),
        "subtype_macro_f1": float(f1_score(subtype_true, subtype_pred, average="macro", zero_division=0)),
    }

@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    logits_binary, logits_subtype = [], []
    labels_binary, labels_subtype = [], []
    patient_ids, files = [], []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        magnification_idx = batch["magnification_index"].to(device, non_blocking=True)

        outputs = model(images, magnification_idx)

        logits_binary.append(outputs["binary_logits"].detach().float().cpu())
        logits_subtype.append(outputs["subtype_logits"].detach().float().cpu())
        labels_binary.append(batch["binary_label"])
        labels_subtype.append(batch["subtype_label"])
        patient_ids.extend(batch["patient_id"])
        files.extend(batch["file_path"])

    binary_probs = torch.softmax(torch.cat(logits_binary), dim=1).numpy()
    subtype_probs = torch.softmax(torch.cat(logits_subtype), dim=1).numpy()
    binary_preds = np.argmax(binary_probs, axis=1)
    subtype_preds = np.argmax(subtype_probs, axis=1)
    binary_true = torch.cat(labels_binary).numpy()
    subtype_true = torch.cat(labels_subtype).numpy()

    return {
        "binary_accuracy": float(accuracy_score(binary_true, binary_preds)),
        "binary_macro_f1": float(f1_score(binary_true, binary_preds, average="macro", zero_division=0)),
        "binary_balanced_accuracy": float(balanced_accuracy_score(binary_true, binary_preds)),
        "binary_mcc": float(matthews_corrcoef(binary_true, binary_preds)),
        "subtype_accuracy": float(accuracy_score(subtype_true, subtype_preds)),
        "subtype_macro_f1": float(f1_score(subtype_true, subtype_preds, average="macro", zero_division=0)),
        "subtype_weighted_f1": float(f1_score(subtype_true, subtype_preds, average="weighted", zero_division=0)),
        "subtype_balanced_accuracy": float(balanced_accuracy_score(subtype_true, subtype_preds)),
        "subtype_mcc": float(matthews_corrcoef(subtype_true, subtype_preds)),
        "binary_confusion_matrix": confusion_matrix(binary_true, binary_preds).tolist(),
        "subtype_confusion_matrix": confusion_matrix(subtype_true, subtype_preds).tolist(),
        "subtype_probs": subtype_probs.tolist(),
        "binary_probs": binary_probs.tolist(),
        "subtype_preds": subtype_preds.tolist(),
        "binary_preds": binary_preds.tolist(),
        "subtype_true": subtype_true.tolist(),
        "binary_true": binary_true.tolist(),
        "patient_ids": patient_ids,
        "files": files,
    }

print("[OK] Training & evaluation routines compiled.")

# Section 11: 5-Fold Patient-Disjoint Cross-Validation Execution

In [ ]:
# ============================================================
# Section 11: 5-Fold Patient-Disjoint Cross-Validation Execution
# ============================================================

def run_fold(fold_idx: int, full_df: pd.DataFrame):
    fold_output_dir = OUTPUT_DIR / f"fold_{fold_idx}"
    fold_output_dir.mkdir(parents=True, exist_ok=True)

    train_idx, test_idx = split_indices[fold_idx]
    train_df_full = full_df.iloc[train_idx].copy().reset_index(drop=True)
    test_df = full_df.iloc[test_idx].copy().reset_index(drop=True)

    # Carve out 10% validation patients from training pool
    train_patients = list(set(train_df_full["patient_id"]))
    val_sample_count = max(1, int(0.10 * len(train_patients)))
    val_patients = set(pd.Series(train_patients).sample(val_sample_count, random_state=CONFIG["seed"] + fold_idx))

    val_df = train_df_full[train_df_full["patient_id"].isin(val_patients)].copy().reset_index(drop=True)
    train_df = train_df_full[~train_df_full["patient_id"].isin(val_patients)].copy().reset_index(drop=True)

    binary_weights, subtype_weights = compute_class_weights(train_df)
    criterion = HierarchicalLoss(binary_weights.to(DEVICE), subtype_weights.to(DEVICE)).to(DEVICE)

    train_loader, val_loader, test_loader = build_dataloaders(train_df, val_df, test_df)

    model = OMNetV3().to(DEVICE)

    base_params = list(model.cnn_backbone.parameters()) + list(model.vit_backbone.parameters())
    head_params = (list(model.cnn_proj.parameters()) + list(model.vit_proj.parameters()) +
                   list(model.fusion.parameters()) + list(model.binary_head.parameters()) +
                   list(model.subtype_head.parameters()))

    optimizer = torch.optim.AdamW([
        {"params": base_params, "lr": CONFIG["base_lr"]},
        {"params": head_params, "lr": CONFIG["head_lr"]},
    ], weight_decay=CONFIG["weight_decay"])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["max_epochs"])

    best_val_f1 = -1.0
    best_state = None
    history = []
    patience_counter = 0

    print(f"\n--- Fold {fold_idx}: Train={len(train_df)} imgs ({train_df['patient_id'].nunique()} pats) | Val={len(val_df)} imgs ({val_df['patient_id'].nunique()} pats) | Test={len(test_df)} imgs ({test_df['patient_id'].nunique()} pats) ---")

    for epoch in range(1, CONFIG["max_epochs"] + 1):
        epoch_start = time.time()

        # Progressive unfreezing: warmup heads first
        if epoch <= CONFIG["warmup_epochs"]:
            set_requires_grad(model.cnn_backbone, False)
            set_requires_grad(model.vit_backbone, False)
        else:
            set_requires_grad(model.cnn_backbone, True)
            set_requires_grad(model.vit_backbone, True)

        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_metrics = evaluate_model(model, val_loader, DEVICE)
        scheduler.step()
        epoch_time = time.time() - epoch_start

        record = {
            "epoch": epoch,
            "train_loss": round(train_metrics["loss"], 4),
            "train_subtype_f1": round(train_metrics["subtype_macro_f1"], 4),
            "val_binary_f1": round(val_metrics["binary_macro_f1"], 4),
            "val_subtype_macro_f1": round(val_metrics["subtype_macro_f1"], 4),
            "val_subtype_acc": round(val_metrics["subtype_accuracy"], 4),
        }
        history.append(record)

        print(f"  Epoch {epoch:2d} | {epoch_time:4.1f}s | Train Loss: {record['train_loss']:.4f} | Val Subtype F1: {record['val_subtype_macro_f1']:.4f} | Val Bin F1: {record['val_binary_f1']:.4f}")

        if val_metrics["subtype_macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["subtype_macro_f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, fold_output_dir / "best_model.pth")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= CONFIG["early_stopping_patience"] and not CONFIG["smoke_test"]:
                print(f"  [INFO] Early stopping triggered at epoch {epoch}.")
                break

    with open(fold_output_dir / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(DEVICE)
    test_metrics = evaluate_model(model, test_loader, DEVICE)

    with open(fold_output_dir / "test_metrics.json", "w") as f:
        json.dump(test_metrics, f, indent=2)

    # Inter-fold memory cleanup (ROCm safety)
    del model, optimizer, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return test_metrics, history

all_fold_results = []
for fold_idx in range(CONFIG["n_splits"]):
    print(f"\n============================================================")
    print(f"  Running Patient-Disjoint Fold {fold_idx + 1}/{CONFIG['n_splits']}")
    print(f"============================================================")
    test_metrics, history = run_fold(fold_idx, metadata)
    all_fold_results.append({
        "fold_idx": fold_idx,
        "test_metrics": test_metrics,
        "history": history,
    })
    print(f"[OK] Fold {fold_idx} finished. Test Subtype Macro-F1: {test_metrics['subtype_macro_f1']:.4f} | Binary Acc: {test_metrics['binary_accuracy']:.4f}")

# Section 12: Cross-Fold Results Aggregation & Statistical Summary

In [ ]:
# ============================================================
# Section 12: Cross-Fold Results Aggregation
# ============================================================

def aggregate_results(results):
    metric_keys = [
        "binary_accuracy", "binary_macro_f1", "binary_balanced_accuracy", "binary_mcc",
        "subtype_accuracy", "subtype_macro_f1", "subtype_weighted_f1", "subtype_balanced_accuracy", "subtype_mcc"
    ]
    summary = {}
    for key in metric_keys:
        values = [r["test_metrics"][key] for r in results]
        values = np.array(values, dtype=float)
        summary[key] = {
            "mean": float(values.mean()),
            "std": float(values.std()),
            "min": float(values.min()),
            "max": float(values.max()),
            "per_fold": values.tolist()
        }
    return summary

aggregated_summary = aggregate_results(all_fold_results)

print("=" * 65)
print("  OMNet-V3 5-Fold Cross-Validation Aggregate Metrics")
print("=" * 65)
for metric, stats in aggregated_summary.items():
    print(f"  {metric:28s}: {stats['mean']*100:.2f}% +/- {stats['std']*100:.2f}% (range: {stats['min']*100:.2f}% - {stats['max']*100:.2f}%)")
print("=" * 65)

# Section 13: Diagnostic Confusion Matrices (Binary & 8-Subtype)

In [ ]:
# ============================================================
# Section 13: Confusion Matrices (Binary & 8-Subtype)
# ============================================================

def plot_confusion_matrix(cm, labels, title, fname):
    fig, ax = plt.subplots(figsize=(7, 6))
    cm_arr = np.array(cm)
    cm_norm = cm_arr.astype(float) / np.maximum(cm_arr.sum(axis=1, keepdims=True), 1)
    labels_annot = np.array([f"{c}\n({p:.1%})" for c, p in zip(cm_arr.flatten(), cm_norm.flatten())]).reshape(cm_arr.shape)
    sns.heatmap(cm_arr, annot=labels_annot, fmt="", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()
    save_figure(fig, fname, dpi=300)
    plt.show()
    plt.close(fig)

cm_binary_total = np.zeros((2, 2), dtype=int)
cm_subtype_total = np.zeros((8, 8), dtype=int)

for res in all_fold_results:
    cm_binary_total += np.array(res["test_metrics"]["binary_confusion_matrix"])
    cm_subtype_total += np.array(res["test_metrics"]["subtype_confusion_matrix"])

plot_confusion_matrix(cm_binary_total, CONFIG["binary_classes"], "Aggregated Binary Confusion Matrix (All Folds)", "confusion_matrix_binary.png")
plot_confusion_matrix(cm_subtype_total, CONFIG["subtype_order"], "Aggregated 8-Subtype Confusion Matrix (All Folds)", "confusion_matrix_8class.png")

# Section 14: ROC & Precision-Recall Curves

In [ ]:
# ============================================================
# Section 14: ROC & Precision-Recall Curves
# ============================================================

all_bin_probs, all_bin_trues = [], []
all_sub_probs, all_sub_trues = [], []

for res in all_fold_results:
    all_bin_probs.extend([p[1] for p in res["test_metrics"]["binary_probs"]])
    all_bin_trues.extend(res["test_metrics"]["binary_true"])
    all_sub_probs.extend(res["test_metrics"]["subtype_probs"])
    all_sub_trues.extend(res["test_metrics"]["subtype_true"])

all_bin_probs = np.array(all_bin_probs)
all_bin_trues = np.array(all_bin_trues)
all_sub_probs = np.array(all_sub_probs)
all_sub_trues = np.array(all_sub_trues)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Binary ROC
fpr, tpr, _ = roc_curve(all_bin_trues, all_bin_probs)
bin_auc = roc_auc_score(all_bin_trues, all_bin_probs)
axes[0].plot(fpr, tpr, label=f"Binary (AUC={bin_auc:.4f})", color="#e74c3c", linewidth=2.5)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("Binary ROC Curve (Aggregated Across Folds)", fontweight="bold")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc="lower right")

# Subtype OvR ROC Curves
for i, subtype in enumerate(CONFIG["subtype_order"]):
    sub_true_binary = (all_sub_trues == i).astype(int)
    if sub_true_binary.sum() > 0:
        sub_fpr, sub_tpr, _ = roc_curve(sub_true_binary, all_sub_probs[:, i])
        sub_auc = roc_auc_score(sub_true_binary, all_sub_probs[:, i])
        axes[1].plot(sub_fpr, sub_tpr, label=f"{subtype} (AUC={sub_auc:.3f})")

axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[1].set_title("Subtype OvR ROC Curves (8 Classes)", fontweight="bold")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
save_figure(fig, "roc_curves.png", dpi=300)
plt.show()
plt.close(fig)

# Section 15: Fusion Gate Alpha Analysis across Magnifications

In [ ]:
# ============================================================
# Section 15: Fusion Gate Analysis across Magnifications
# ============================================================

best_fold_model = OMNetV3().to(DEVICE)
best_fold_ckpt = torch.load(OUTPUT_DIR / "fold_0" / "best_model.pth", map_location=DEVICE, weights_only=False)
best_fold_model.load_state_dict(best_fold_ckpt)
best_fold_model.eval()

_, _, test_loader_f0 = build_dataloaders(
    metadata.iloc[split_indices[0][0]],
    metadata.iloc[split_indices[0][0][:10]],
    metadata.iloc[split_indices[0][1]]
)

alpha_by_mag = {m: [] for m in CONFIG["magnification_levels"]}
with torch.no_grad():
    for batch in test_loader_f0:
        images = batch["image"].to(DEVICE, non_blocking=True)
        mag_idx = batch["magnification_index"].to(DEVICE, non_blocking=True)
        outputs = best_fold_model(images, mag_idx)
        alphas = outputs["alpha"].detach().cpu().flatten().numpy()
        for i, m_idx in enumerate(batch["magnification_index"].numpy()):
            mag_val = CONFIG["magnification_levels"][m_idx]
            alpha_by_mag[mag_val].append(alphas[i])

print("=" * 65)
print("  Fusion Gate (Alpha) Distribution by Magnification")
print("=" * 65)
for mag, alphas in alpha_by_mag.items():
    if alphas:
        print(f"  {mag:3d}X: mean alpha = {np.mean(alphas):.4f} +/- {np.std(alphas):.4f} (Weight on CNN branch)")
print("=" * 65)

fig, ax = plt.subplots(figsize=(8, 5))
data_for_plot = [[mag, a] for mag, alphas in alpha_by_mag.items() for a in alphas]
df_gate = pd.DataFrame(data_for_plot, columns=["Magnification", "Alpha"])
sns.boxplot(data=df_gate, x="Magnification", y="Alpha", ax=ax, palette="Blues")
ax.set_title("Adaptive Fusion Gate Alpha across Magnification Levels (40X - 400X)", fontweight="bold")
ax.set_ylabel("Alpha (CNN weight vs ViT weight)")
plt.tight_layout()
save_figure(fig, "fusion_gate_analysis.png", dpi=300)
plt.show()
plt.close(fig)

# Section 16: t-SNE 2D Embedding Visualization of Fused Representations

In [ ]:
# ============================================================
# Section 16: t-SNE Embedding Visualization
# ============================================================

all_features, all_sub_lbls = [], []
with torch.no_grad():
    for batch in test_loader_f0:
        images = batch["image"].to(DEVICE, non_blocking=True)
        mag_idx = batch["magnification_index"].to(DEVICE, non_blocking=True)
        outputs = best_fold_model(images, mag_idx)
        all_features.append(outputs["f_out"].cpu().numpy())
        all_sub_lbls.extend(batch["subtype_label"].numpy())

all_features = np.concatenate(all_features, axis=0)
all_sub_lbls = np.array(all_sub_lbls)

tsne = TSNE(n_components=2, random_state=CONFIG["seed"], perplexity=min(30, max(5, len(all_features)-1)))
emb_2d = tsne.fit_transform(all_features)

fig, ax = plt.subplots(figsize=(9, 8))
for i, subtype in enumerate(CONFIG["subtype_order"]):
    mask = (all_sub_lbls == i)
    if mask.sum() > 0:
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], label=subtype, alpha=0.7, s=40)
ax.set_title("t-SNE Projection of OMNet-V3 Fused Feature Embeddings by Subtype", fontweight="bold")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
save_figure(fig, "tsne_embeddings.png", dpi=300)
plt.show()
plt.close(fig)

# Section 17: Experiment Summary & Metrics Export

In [ ]:
# ============================================================
# Section 17: Experiment Summary & Export
# ============================================================

experiment_record = {
    "timestamp": datetime.now().isoformat(),
    "hardware": {
        "device": "AMD Radeon RX 9060 XT 16GB",
        "backend": "ROCm / HIP",
        "precision": "FP32"
    },
    "config": CONFIG,
    "aggregated_results": aggregated_summary,
    "folds_completed": len(all_fold_results),
}

with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump(experiment_record, f, indent=2, default=str)

df_summary = pd.DataFrame(aggregated_summary).T
df_summary.to_csv(OUTPUT_DIR / "metrics_summary.csv")

print("=" * 75)
print("  OMNet-V3 Final Execution Summary")
print("=" * 75)
print(f"  Dataset             : BreakHis ({CONFIG['dataset_id']})")
print(f"  Magnifications      : {CONFIG['magnification_levels']}")
print(f"  Folds Evaluated     : {CONFIG['n_splits']}")
print(f"  Subtype Macro-F1    : {aggregated_summary['subtype_macro_f1']['mean']*100:.2f}% +/- {aggregated_summary['subtype_macro_f1']['std']*100:.2f}%")
print(f"  Binary Accuracy     : {aggregated_summary['binary_accuracy']['mean']*100:.2f}% +/- {aggregated_summary['binary_accuracy']['std']*100:.2f}%")
print(f"  Output Directory    : {OUTPUT_DIR}")
print("=" * 75)
print("[OK] All fold checkpoints, metrics summary, and publishable figures saved successfully.")